# Load Professor Reviews into Pinecone

This notebook loads professor review data into a Pinecone vector database.
We use HuggingFace embeddings to create vector representations of each review,
then upsert them into Pinecone for semantic search.

In [ ]:
# Install dependencies
!pip install pinecone-client openai python-dotenv

In [ ]:
import json
import os
from pinecone import Pinecone, ServerlessSpec

# Initialize Pinecone
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

# Create index if it doesn't exist
index_name = "rag"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=768,  # HuggingFace embedding dimension
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)
print(f"Index '{index_name}' ready!")

In [ ]:
# Load the reviews data
with open("data/reviews.json", "r") as f:
    data = json.load(f)

reviews = data["reviews"]
print(f"Loaded {len(reviews)} professor reviews")

In [ ]:
from huggingface_hub import InferenceClient

# Create embeddings using HuggingFace
hf_client = InferenceClient(token=os.environ.get("HUGGINGFACE_API_KEY"))

vectors = []
for i, review in enumerate(reviews):
    text = f"Professor: {review['professor']}. Subject: {review['subject']}. Rating: {review['stars']} stars. Review: {review['review']}"
    
    # Generate embedding
    embedding = hf_client.feature_extraction(
        text,
        model="sentence-transformers/all-MiniLM-L6-v2"
    )
    
    # Flatten if needed (some models return nested arrays)
    if hasattr(embedding, 'tolist'):
        embedding = embedding.tolist()
    if isinstance(embedding[0], list):
        embedding = embedding[0]
    
    vectors.append({
        "id": f"review-{i}",
        "values": embedding,
        "metadata": {
            "professor": review["professor"],
            "subject": review["subject"],
            "stars": review["stars"],
            "review": review["review"]
        }
    })

print(f"Created {len(vectors)} embeddings")

In [ ]:
# Upsert vectors into Pinecone
index.upsert(vectors=vectors)

# Check stats
stats = index.describe_index_stats()
print(f"Pinecone index stats: {stats}")
print("\nData loaded successfully!")